In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import os

In [2]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# Define constants
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 30

In [4]:
# Paths to your data directories (using your specified paths)
tea_leaf_dir = r"D:\Agro_model\training\model01\Binary_tea_nonTea\Tea_leaf_images_B"
non_tea_leaf_dir = r"D:\Agro_model\training\model01\Binary_tea_nonTea\Non-tea_leaf_images_B"


In [5]:
# Check if directories exist
print(f"Tea leaf directory exists: {os.path.exists(tea_leaf_dir)}")
print(f"Non-tea leaf directory exists: {os.path.exists(non_tea_leaf_dir)}")

Tea leaf directory exists: True
Non-tea leaf directory exists: True


In [7]:
# Count images in each directory
tea_leaf_count = len([f for f in os.listdir(tea_leaf_dir) if os.path.isfile(os.path.join(tea_leaf_dir, f))])
non_tea_leaf_count = len([f for f in os.listdir(non_tea_leaf_dir) if os.path.isfile(os.path.join(non_tea_leaf_dir, f))])

print(f"Number of tea leaf images: {tea_leaf_count}")
print(f"Number of non-tea leaf images: {non_tea_leaf_count}")

Number of tea leaf images: 6000
Number of non-tea leaf images: 6000


In [8]:
# Set up data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

In [9]:
parent_dir = r"D:\Agro_model\training\model01\Binary_tea_nonTea"

# Training generator
train_generator = train_datagen.flow_from_directory(
    parent_dir,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',  # binary classification
    classes=['Tea_leaf_images_B', 'Non-tea_leaf_images_B'],  # Specify class folder names
    subset='training',
    shuffle=True
)

# Validation generator
validation_generator = train_datagen.flow_from_directory(
    parent_dir,
    target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',  # binary classification
    classes=['Tea_leaf_images_B', 'Non-tea_leaf_images_B'],  # Specify class folder names
    subset='validation',
    shuffle=False
)

print(f"Class indices: {train_generator.class_indices}")

Found 9600 images belonging to 2 classes.
Found 2400 images belonging to 2 classes.
Class indices: {'Tea_leaf_images_B': 0, 'Non-tea_leaf_images_B': 1}


In [10]:
# Create a binary classifier model
def create_binary_classifier():
    model = Sequential([
        # First Convolutional Block
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), padding='same'),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Second Convolutional Block
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Third Convolutional Block
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(0.25),
        
        # Flatten and Dense Layers
        Flatten(),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.5),
        Dense(1, activation='sigmoid')  # Binary output - tea leaf or not
    ])
    
    # Compile the model
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Create and display the model
binary_model = create_binary_classifier()
binary_model.summary()


C:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 256, 256, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256, 256, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 256, 256, 32)        │           9,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 256, 256, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 128, 128, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128, 128, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 128, 128, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 128, 128, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 128, 128, 64)        │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_3                │ (None, 128, 128, 64)        │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 64, 64, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64, 64, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 64, 64, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_4                │ (None, 64, 64, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 32, 32, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 32, 32, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 131072)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │      33,554,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_5                │ (None, 256)                 │           1,0

 Total params: 33,696,673 (128.54 MB)

 Trainable params: 33,695,521 (128.54 MB)

 Non-trainable params: 1,152 (4.50 KB)

In [11]:
# Set up callbacks
checkpoint = ModelCheckpoint(
    'tea_leaf_binary_classifier.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

callbacks = [checkpoint, early_stopping, reduce_lr]

# Calculate steps per epoch and validation steps
steps_per_epoch = train_generator.samples // BATCH_SIZE
validation_steps = validation_generator.samples // BATCH_SIZE

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Validation steps: {validation_steps}")

Steps per epoch: 300
Validation steps: 75


In [12]:
# Train the binary classifier
history = binary_model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Save the final model
binary_model.save('tea_leaf_binary_classifier_final.h5')


C:\Users\User\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9656 - loss: 0.1014   
Epoch 1: val_accuracy improved from -inf to 0.76458, saving model to tea_leaf_binary_classifier.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 749s 2s/step - accuracy: 0.9656 - loss: 0.1013 - val_accuracy: 0.7646 - val_loss: 0.5864 - learning_rate: 0.0010
Epoch 2/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9928 - loss: 0.0255   
Epoch 2: val_accuracy improved from 0.76458 to 0.76625, saving model to tea_leaf_binary_classifier.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 702s 2s/step - accuracy: 0.9928 - loss: 0.0255 - val_accuracy: 0.7663 - val_loss: 1.1512 - learning_rate: 0.0010
Epoch 3/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9933 - loss: 0.0190   
Epoch 3: val_accuracy did not improve from 0.76625
300/300 ━━━━━━━━━━━━━━━━━━━━ 703s 2s/step - accuracy: 0.9933 - loss: 0.0190 - val_accuracy: 0.7400 - val_loss: 2.4271 - learning_rate: 0.0010
Epoch 4/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9922 - loss: 0.0243   
Epoch 4: val_accuracy did not improve from 0.76625
300/300 ━━━━━━━━━━━━━━━━━━━━ 704s 2s/step - accuracy: 0.9922 - loss: 0.0242 - val_accuracy: 0.7588 - val_loss: 2.9977 - learning_rate: 0.0010
Epoch 5/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9941 - loss: 0.0221   
Epoch 5: val_accuracy did not improve from 0.76625
300/300 ━━━━━━━━━━━━━━━━━━━━ 704s 2s/step - accuracy: 0.9941 - loss: 0.0221 - val_accuracy: 0.6967 - val_loss: 2.1493 - learning_rate: 0.0010
Epoch 6/30
300/

300/300 ━━━━━━━━━━━━━━━━━━━━ 704s 2s/step - accuracy: 0.9982 - loss: 0.0048 - val_accuracy: 0.7742 - val_loss: 1.4617 - learning_rate: 2.0000e-04
Epoch 9/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9987 - loss: 0.0042   
Epoch 9: val_accuracy improved from 0.77417 to 0.77542, saving model to tea_leaf_binary_classifier.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 705s 2s/step - accuracy: 0.9987 - loss: 0.0042 - val_accuracy: 0.7754 - val_loss: 1.3061 - learning_rate: 2.0000e-04
Epoch 10/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9977 - loss: 0.0051   
Epoch 10: val_accuracy improved from 0.77542 to 0.79625, saving model to tea_leaf_binary_classifier.h5


300/300 ━━━━━━━━━━━━━━━━━━━━ 703s 2s/step - accuracy: 0.9977 - loss: 0.0051 - val_accuracy: 0.7962 - val_loss: 0.9578 - learning_rate: 2.0000e-04
Epoch 11/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9989 - loss: 0.0031   
Epoch 11: val_accuracy did not improve from 0.79625

Epoch 11: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
300/300 ━━━━━━━━━━━━━━━━━━━━ 703s 2s/step - accuracy: 0.9989 - loss: 0.0031 - val_accuracy: 0.7200 - val_loss: 1.3411 - learning_rate: 2.0000e-04
Epoch 11: early stopping
Restoring model weights from the end of the best epoch: 1.
